In [ ]:
# ================================
# 📌 Full PhonePe Pulse ETL Script
# ================================

# 1️⃣ Install libraries
import sys
!{sys.executable} -m pip install pandas psycopg2-binary gitpython

# 2️⃣ Import required modules
import os
import json
import pandas as pd
from git import Repo
import psycopg2
from psycopg2.extras import execute_values

# 3️⃣ Clone GitHub data if not exists
repo_url = "https://github.com/PhonePe/pulse.git"
clone_path = "Phonepe_clonedata"

if not os.path.exists(clone_path):
    Repo.clone_from(repo_url, clone_path)
    print("✅ Data cloned successfully")
else:
    print("⚠️ Folder already exists – skipping clone")

# 4️⃣ Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    user="postgres",
    password="qwerty123",
    port="5432"
)
conn.autocommit=True
cursor = conn.cursor()

# Create database if not exists
cursor.execute("SELECT 1 FROM pg_database WHERE datname='phonepe_db'")
if not cursor.fetchone():
    cursor.execute("CREATE DATABASE phonepe_db")
    print("✅ Database created")
else:
    print("⚠️ Database already exists")

# Connect to project database
conn = psycopg2.connect(
    host="localhost",
    user="postgres",
    password="qwerty123",
    database="phonepe_db",
    port="5432"
)
cursor = conn.cursor()
print("✅ Connected to phonepe_db")

# 5️⃣ Create 9 tables
create_tables = """
CREATE TABLE IF NOT EXISTS aggregated_transaction(
state TEXT, year INT, quarter INT,
transaction_type TEXT,
transaction_count BIGINT,
transaction_amount FLOAT
);

CREATE TABLE IF NOT EXISTS aggregated_user(
state TEXT, year INT, quarter INT,
user_type TEXT,
user_count BIGINT
);

CREATE TABLE IF NOT EXISTS aggregated_insurance(
state TEXT, year INT, quarter INT,
insurance_type TEXT,
insurance_count BIGINT,
insurance_amount FLOAT
);

CREATE TABLE IF NOT EXISTS map_transaction(
state TEXT, year INT, quarter INT,
district TEXT,
transaction_count BIGINT,
transaction_amount FLOAT
);

CREATE TABLE IF NOT EXISTS map_user(
state TEXT, year INT, quarter INT,
district TEXT,
registered_users BIGINT,
app_opens BIGINT
);

CREATE TABLE IF NOT EXISTS map_insurance(
state TEXT, year INT, quarter INT,
district TEXT,
insurance_count BIGINT,
insurance_amount FLOAT
);

CREATE TABLE IF NOT EXISTS top_transaction(
state TEXT, year INT, quarter INT,
district TEXT,
transaction_count BIGINT,
transaction_amount FLOAT
);

CREATE TABLE IF NOT EXISTS top_user(
state TEXT, year INT, quarter INT,
district TEXT,
registered_users BIGINT
);

CREATE TABLE IF NOT EXISTS top_insurance(
state TEXT, year INT, quarter INT,
district TEXT,
insurance_count BIGINT,
insurance_amount FLOAT
);
"""
cursor.execute(create_tables)
conn.commit()
print("✅ All 9 tables created")

# 6️⃣ Bulk insert function
def load_table(df, table, columns):
    if not df.empty:
        query = f"INSERT INTO {table} ({columns}) VALUES %s"
        execute_values(cursor, query, df.values.tolist())
        conn.commit()
        print(f"✅ {table} loaded with {len(df)} rows")
    else:
        print(f"⚠️ No data to load for {table}")

# 7️⃣ Generic function to read JSON files recursively
def extract_data(base_path, table_type):
    data = []
    for state in os.listdir(base_path):
        state_path = os.path.join(base_path, state)
        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)
            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    file_path = os.path.join(year_path, file)
                    with open(file_path) as f:
                        j = json.load(f)
                        try:
                            if table_type == "aggregated_transaction":
                                for t in j["data"]["transactionData"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        t["name"],
                                        t["paymentInstruments"][0]["count"],
                                        t["paymentInstruments"][0]["amount"]
                                    ])
                            elif table_type == "aggregated_user":
                                for u in j["data"]["usersByDevice"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        u["brand"], u["count"]
                                    ])
                            elif table_type == "aggregated_insurance":
                                for t in j["data"]["transactionData"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        t["name"],
                                        t["paymentInstruments"][0]["count"],
                                        t["paymentInstruments"][0]["amount"]
                                    ])
                            elif table_type == "map_transaction":
                                for d in j["data"]["hoverDataList"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d["name"],
                                        d["metric"][0]["count"],
                                        d["metric"][0]["amount"]
                                    ])
                            elif table_type == "map_user":
                                for d,v in j["data"]["hoverData"].items():
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d, v["registeredUsers"], v["appOpens"]
                                    ])
                            elif table_type == "map_insurance":
                                for d in j["data"]["hoverDataList"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d["name"],
                                        d["metric"][0]["count"],
                                        d["metric"][0]["amount"]
                                    ])
                            elif table_type == "top_transaction":
                                for d in j["data"]["districts"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d["entityName"],
                                        d["metric"]["count"],
                                        d["metric"]["amount"]
                                    ])
                            elif table_type == "top_user":
                                for d in j["data"]["districts"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d["name"],
                                        d["registeredUsers"]
                                    ])
                            elif table_type == "top_insurance":
                                for d in j["data"]["districts"]:
                                    data.append([
                                        state, int(year), int(file.replace(".json","")),
                                        d["entityName"],
                                        d["metric"]["count"],
                                        d["metric"]["amount"]
                                    ])
                        except:
                            pass
    return pd.DataFrame(data)

# 8️⃣ Load all 9 datasets
datasets = {
    "aggregated_transaction": ["state","year","quarter","transaction_type","transaction_count","transaction_amount"],
    "aggregated_user": ["state","year","quarter","user_type","user_count"],
    "aggregated_insurance": ["state","year","quarter","insurance_type","insurance_count","insurance_amount"],
    "map_transaction": ["state","year","quarter","district","transaction_count","transaction_amount"],
    "map_user": ["state","year","quarter","district","registered_users","app_opens"],
    "map_insurance": ["state","year","quarter","district","insurance_count","insurance_amount"],
    "top_transaction": ["state","year","quarter","district","transaction_count","transaction_amount"],
    "top_user": ["state","year","quarter","district","registered_users"],
    "top_insurance": ["state","year","quarter","district","insurance_count","insurance_amount"]
}

base_paths = {
    "aggregated_transaction": "Phonepe_clonedata/data/aggregated/transaction/country/india/state/",
    "aggregated_user": "Phonepe_clonedata/data/aggregated/user/country/india/state/",
    "aggregated_insurance": "Phonepe_clonedata/data/aggregated/insurance/country/india/state/",
    "map_transaction": "Phonepe_clonedata/data/map/transaction/hover/country/india/state/",
    "map_user": "Phonepe_clonedata/data/map/user/hover/country/india/state/",
    "map_insurance": "Phonepe_clonedata/data/map/insurance/hover/country/india/state/",
    "top_transaction": "Phonepe_clonedata/data/top/transaction/country/india/state/",
    "top_user": "Phonepe_clonedata/data/top/user/country/india/state/",
    "top_insurance": "Phonepe_clonedata/data/top/insurance/country/india/state/"
}

for table, cols in datasets.items():
    print(f"⏳ Loading {table} ...")
    df = extract_data(base_paths[table], table)
    load_table(df, table, ",".join(cols))


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


⚠️ Folder already exists – skipping clone
⚠️ Database already exists
✅ Connected to phonepe_db
✅ All 9 tables created
⏳ Loading aggregated_transaction ...
✅ aggregated_transaction loaded with 5034 rows
⏳ Loading aggregated_user ...
✅ aggregated_user loaded with 6732 rows
⏳ Loading aggregated_insurance ...
✅ aggregated_insurance loaded with 682 rows
⏳ Loading map_transaction ...
✅ map_transaction loaded with 20604 rows
⏳ Loading map_user ...
✅ map_user loaded with 20608 rows
⏳ Loading map_insurance ...
✅ map_insurance loaded with 13876 rows
⏳ Loading top_transaction ...
✅ top_transaction loaded with 8296 rows
⏳ Loading top_user ...
✅ top_user loaded with 8296 rows
⏳ Loading top_insurance ...
✅ top_insurance loaded with 5608 rows
